# HumanEval EDA

- 출처 : https://huggingface.co/datasets/openai/openai_humaneval   
- 설명 : OpenAI에서 제공하는 HumanEval 데이터 세트(2021)는 164개의 Python 코딩 문제로 구성되어 있으며, 각 문제는 프롬프트, 예제 case, test case를 포함하고 있으다.(사람이 직접 작성함)
- pass@k

<details>
<summary><strong>상세 분석 및 정리</strong></summary>
노션에 데이터분석으로 정리해둠

1. 문제 구성   

- **문제 구성:** 164개의 손으로 직접 작성된(hand-written) 파이썬 프로그래밍 문제.   
- **내용:** 함수 시그니처, 독스트링(docstring), 문제 설명으로 구성된 프롬프트를 보고, 함수 본문을 완성하는 방식.   
- **평가 방식:** 생성된 코드가 단위 테스트(Unit Tests)를 통과하는지 확인 (평균적으로 문제당 8개의 테스트 케이스 적용)   
   


2. 평가 메트릭(pass@k):   
```bash
pass@k = c/n * sum(1 - comb(n - c, k) / comb(n, k))
```   
각 문제마다 k개의 답을 생성했을 때, 그 중 하나라도 정답이면 성공으로 보는 metric.   
즉, 하나의 문제에서 n개의 답을 생성했고, 그중 c개가 정답일 때, 그 n개 중에서 *중복 없이* k개를 랜덤하게 뽑았을 때 최소 하나가 정답일 확률   


3. 한계점   
파이썬으로 구성되어있으며, 난이도가 낮은 알고리즘 문제 중심이다.   
HumanEval-V와 같은 다중 모달 모델 평가를 위한 복잡한 다이어그램을 활용하는 확장 버전도 개발되고 있다.

</details>

## overview :
- ch01 : 데이터 톺아보기
- ch02 : task 활용 관점으로 데이터 분석

---
## ch01 : 데이터 톺아보기   

01. 데이터 확인
02. 한 샘플 자세히 보기
03. **동작 흐름 관점 분석**
04. 연구 관점 정리

### 1. 데이터 확인하기

In [1]:
# 1. 데이터 확인
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path("../../datasets/humaneval/raw/humaneval.jsonl")

rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head()

,task_id,prompt,canonical_solution,test,entry_point
0,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,"for idx, elem in enumerate(numbers):\n ...","\n\nMETADATA = {\n 'author': 'jt',\n 'da...",has_close_elements
1,HumanEval/1,from typing import List\n\n\ndef separate_pare...,result = []\n current_string = []\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da...",separate_paren_groups
2,HumanEval/2,\n\ndef truncate_number(number: float) -> floa...,return number % 1.0\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'da...",truncate_number
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,balance = 0\n\n for op in operations:\n...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da...",below_zero
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean = sum(numbers) / len(numbers)\n re...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da...",mean_absolute_deviation


In [2]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.dtypes

shape: (164, 5)
columns: ['task_id', 'prompt', 'canonical_solution', 'test', 'entry_point']


task_id               object
prompt                object
canonical_solution    object
test                  object
entry_point           object
dtype: object

### 1-2. 데이터 필드 확인하기   
출처 : https://huggingface.co/datasets/openai/openai_humaneval   

> Data Fields:   
> - task_id
> - prompt : llm의 프롬프트에 넣어지는, 모델 입력값
> - canonical_solution : 정답 참고용(body-only)
> - test : 테스트하기 위한 함수 정의
> - entry_point : 테스트 진입점, 함수명

### 2. 샘플 데이터 자세히 뜯어보기

In [3]:
# 샘플 1개에 대해 확인하기
sample = df.iloc[0].to_dict()
for k, v in sample.items():
    print(f"\n===== {k} =====")
    print(v if isinstance(v, str) else repr(v))


===== task_id =====
HumanEval/0

===== prompt =====
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


===== canonical_solution =====
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False


===== test =====


METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True
    assert candi

### 3. 동작 흐름 관점 분석

한 샘플의 동작 :    
(1) input : prompt  

(2) output y : 모델이 함수 body 생성  
    → code = prompt + y  

(3) exec(code)    
    → has_close_elements 함수 생성 (candidate)

(4) exec(test)  
    → check 함수 생성  

(5) check(has_close_elements)     
    → assert로 평가


(1) prompt   
**모델 입력값 :  함수 틀이 입력되어 있음**
> 함수이름과 파라미터, docstring(주석)을 포함함.   

In [4]:
# ===== prompt =====
# from typing import List


# def has_close_elements(numbers: List[float], threshold: float) -> bool:
#     """ Check if in given list of numbers, are any two numbers closer to each other than
#     given threshold.
#     >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
#     False
#     >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
#     True
#     """

(2) output   
**모델 출력**
> string(문자열) 형태로 출력함.   

In [5]:
y = """
def has_close_elements(numbers, threshold):
    return True
"""

(3) exec(code)
exec는 문자열로 된 코드(code)를 실제 파이썬 코드로 실행하는 함수로, 파이썬에 built-in함수
   
모델의 출력(문자열)을 함수 형태 code로 string을 합침. 아직 함수로 정의된 것은 아님. 그래서 exec(code)를 실행하면 문자열 안의 코드가 실제로 실행됨.   
이것이 executer로 작동함.   
- 이때, executoer가 작동할 수 없는 에러가 발생하면 structaul

(4) exec(test)   
check()를 함수로 만드는 과정.    
    
(5) check(모델이 만든 함수)를 실행하면 예제들 넣어서 정답확인

In [6]:
"""
===== test =====


METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False
    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 1.0) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 0.5) == False

"""

"\n===== test =====\n\n\nMETADATA = {\n    'author': 'jt',\n    'dataset': 'test'\n}\n\n\ndef check(candidate):\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True\n    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False\n    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1) == True\n    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 1.0) == True\n    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 0.5) == False\n\n"

### 4. 연구 포인트

이 흐름 때문에 sLM으로 작업하면 executer 오류(structual)가 발견되었었다.    
- 들여쓰기 오류 (indentation)
- 함수 밖 코드 생성
- return 누락
- etc ..   

LM언어모델은 자율성 높게 출력이 나오기 때문에 이를 제어하는 작업이 필요함.   
하드코딩으로 자연어 처리를 하거나(re라이브럴리) 등의 스킬이 필요할 것으로 보임.

>> sLM 출력의 structural failure를 완화하는 normalization/validation layer 필요  
sLM - (여기) - exaluator

In [7]:
"""
1차 버전 아이디어 

대상: humaneval, mbpp, humaneval_pro, mbpp_pro
입력: prompt, raw_output, entry_point
출력: normalized_code, body_only, diagnostics
핵심 규칙:
코드 블록 추출
함수명 일치 확인
body indentation 복구
함수 밖 텍스트 제거
실행 전 ast.parse / compile 기반 문법 점검
실패 시 “복구 불가”로 명시


구조
code-output-normalizer/
├── function_task/
│   ├── extract
│   ├── repair
│   └── validate
├── patch_task/
│   ├── extract
│   ├── repair
│   └── validate
└── shared/
    ├── diagnostics
    └── status taxonomy


A. function-task normalizer
대상:
humaneval
mbpp
humaneval_pro
mbpp_pro


B. patch-task normalizer
대상:
swebench_lite

"""

'\n1차 버전 아이디어 \n\n대상: humaneval, mbpp, humaneval_pro, mbpp_pro\n입력: prompt, raw_output, entry_point\n출력: normalized_code, body_only, diagnostics\n핵심 규칙:\n코드 블록 추출\n함수명 일치 확인\nbody indentation 복구\n함수 밖 텍스트 제거\n실행 전 ast.parse / compile 기반 문법 점검\n실패 시 “복구 불가”로 명시\n\n\n구조\ncode-output-normalizer/\n├── function_task/\n│   ├── extract\n│   ├── repair\n│   └── validate\n├── patch_task/\n│   ├── extract\n│   ├── repair\n│   └── validate\n└── shared/\n    ├── diagnostics\n    └── status taxonomy\n\n\nA. function-task normalizer\n대상:\nhumaneval\nmbpp\nhumaneval_pro\nmbpp_pro\n\n\nB. patch-task normalizer\n대상:\nswebench_lite\n\n'

---
## ch02 : 데이터 톺아보기   

01. Input / Output / Evaluation 길이 분석
02. Prompt 구조 분석 (input format)
03. Test 구조 분석 (constraint structure)
04. Research implication 

### 1. 길이 관점

In [8]:
# prompt_len : 모델에 들어가는 길이
# canonical_solution_len : 모델이 출력하는 길이 관점 추정
# test_len : 평가 복잡도? 검증 조건의 길이

for col in ["prompt", "canonical_solution", "test"]:
    df[f"{col}_len"] = df[col].astype(str).str.len()

df[["prompt_len", "canonical_solution_len", "test_len"]].describe()

,prompt_len,canonical_solution_len,test_len
count,164.000000,164.000000,164.000000
mean,450.597561,180.865854,509.109756
std,230.303766,139.527974,279.919182
min,115.000000,16.000000,117.000000
25%,287.750000,78.000000,297.250000
50%,396.000000,148.500000,442.500000
75%,568.000000,241.750000,675.000000
max,1360.000000,864.000000,1804.000000


In [9]:
preview_cols = ["task_id", "entry_point", "prompt_len", "canonical_solution_len", "test_len"]
df[preview_cols].head(10)

,task_id,entry_point,prompt_len,canonical_solution_len,test_len
0,HumanEval/0,has_close_elements,348,252,531
1,HumanEval/1,separate_paren_groups,506,419,441
2,HumanEval/2,truncate_number,331,24,212
3,HumanEval/3,below_zero,448,131,394
4,HumanEval/4,mean_absolute_deviation,430,101,275
5,HumanEval/5,intersperse,287,192,234
6,HumanEval/6,parse_nested_parens,436,336,261
7,HumanEval/7,filter_by_substring,330,50,443
8,HumanEval/8,sum_product,372,140,300
9,HumanEval/9,rolling_max,288,237,279


### 2. prompt 구조 관찰

In [10]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'prompt'])}\n")
    print(df.loc[i, "prompt"])


================ SAMPLE 0 ================ len : 348

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """


================ SAMPLE 1 ================ len : 506

from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and return the list of those.
    Separate groups are balanced (each open brace is properly closed) and not nested within each other
    Ignore any spaces in the input string.
    >>> separate_paren_groups('( ) (( )) (( )( ))')
    ['()', '(())', '(()())']
    """


================ SAMPLE 2 =======

In [11]:
import re
def contains_import(text):
    return "import " in str(text) or "from " in str(text)

def contains_doctest(text):
    return ">>>" in str(text)

def contains_docstring(text):
    text = str(text)
    return '"""' in text or "'''" in text

def contains_def(text):
    return re.search(r"def\s+\w+\s*\(", str(text)) is not None

df["prompt_has_import"] = df["prompt"].apply(contains_import)
df["prompt_has_doctest"] = df["prompt"].apply(contains_doctest)
df["prompt_has_docstring"] = df["prompt"].apply(contains_docstring)
df["prompt_has_def"] = df["prompt"].apply(contains_def)

df[[
    "prompt_has_import",
    "prompt_has_doctest",
    "prompt_has_docstring",
    "prompt_has_def"
]].mean()

prompt_has_import       0.225610
prompt_has_doctest      0.463415
prompt_has_docstring    1.000000
prompt_has_def          1.000000
dtype: float64

입력 prompt 분석 결과 :   
- 데이터셋의 prompt는 고정된 템플릿임.   
- doscstring과 def가 모두 있음 : 프롬프트에 설명과 함수틀을 미리 제공함

### 3. test 구조 관찰

In [12]:
for i in range(3):
    print(f"\n================ TEST {i} - {df.loc[i, 'task_id']} ================ len : {len(df.loc[i, 'test'])}\n")
    print(df.loc[i, "test"])


================ TEST 0 - HumanEval/0 ================ len : 531



METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True
    assert candidate([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True
    assert candidate([1.0, 2.0, 5.9, 4.0, 5.0], 0.8) == False
    assert candidate([1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 1.0) == True
    assert candidate([1.1, 2.2, 3.1, 4.1, 5.1], 0.5) == False



================ TEST 1 - HumanEval/1 ================ len : 441



METADATA = {
    'author': 'jt',
    'dataset': 'test'
}


def check(candidate):
    assert candidate('(()()) ((())) () ((())()())') == [
        '(()())', '((()))', '()', '((())()())'
    ]
    assert candidate('() (()) ((())) (((())))') == [
        '()', '(())', '((()))', '(((())))'
    ]
    assert candidate('(()(())((())))') 

In [13]:
def count_asserts(text):
    return len(re.findall(r"\bassert\b", str(text)))

def has_check_function(text):
    return re.search(r"def\s+check\s*\(", str(text)) is not None

df["test_assert_count"] = df["test"].apply(count_asserts)
df["test_has_check_function"] = df["test"].apply(has_check_function)

df[["test_assert_count", "test_has_check_function"]].describe()

,test_assert_count
count,164.000000
mean,8.079268
std,4.537485
min,1.000000
25%,5.000000
50%,7.000000
75%,10.000000
max,26.000000


test관점 : 
- assert 수 = constraint 수로 이해할 수 있음. 많을수록 어렵다고 볼 수 있음

### 4. 연구관점 정리

HumanEval은 '코드를 생성하는 문제'가 아니라 '구조를 유지하면서 constraint를 만족하는 함수 생성 문제'다